In [1]:
from  datetime import datetime, timedelta
import gc
import numpy as np, pandas as pd
import lightgbm as lgb

#Hiển thị max 50 cột
pd.options.display.max_columns = 50

# Trỏ đường dẫn để gọi file trong thư mục src
import sys, os
sys.path.append(os.path.abspath('../src'))

from dataPreprocessing import create_dt, create_fea

In [2]:
#Định nghĩa dữ liệu cho từng cột phân loại
CAL_DTYPES={"event_name_1": "category", "event_name_2": "category", "event_type_1": "category", 
         "event_type_2": "category", "weekday": "category", 'wm_yr_wk': 'int16', "wday": "int16",
        "month": "int16", "year": "int16", "snap_CA": "float32", 'snap_TX': 'float32', 'snap_WI': 'float32' }

#Định nghĩa dữ liệu cho câc cột lien quan đến giá
PRICE_DTYPES = {"store_id": "category", "item_id": "category", "wm_yr_wk": "int16","sell_price":"float32" }

#Ngày bắt đầu lấy dữ liệu
FIRST_DAY = 350

#Định nghĩa một số biến đầu vào
h = 28 
max_lags = 57
tr_last = 1913
fday = datetime(2016,4, 25) 
fday

datetime.datetime(2016, 4, 25, 0, 0)

In [3]:
#Convert dữ liệu sang dạng long
df = create_dt(is_train=True, first_day= FIRST_DAY, nrows=None, price_dt = PRICE_DTYPES, cal_dt = CAL_DTYPES)
df.shape

(40718219, 22)

In [4]:
#Load phân loại deamand đã có từ trước
df_demandType = pd.read_csv("../dataset/result/demand_classification.csv")
df_demandType.head()

,id,ADI,CV2,Category
0,FOODS_1_001_CA_1_validation,2.29,0.26,Intermittent (Ngắt quãng)
1,FOODS_1_001_CA_2_validation,1.93,0.46,Intermittent (Ngắt quãng)
2,FOODS_1_001_CA_3_validation,2.23,0.50,Lumpy (Cục bộ)
3,FOODS_1_001_CA_4_validation,3.87,0.24,Intermittent (Ngắt quãng)
4,FOODS_1_001_TX_1_validation,3.06,0.29,Intermittent (Ngắt quãng)


In [5]:
#Add demand feature vào df
def add_demandFeature(demand_dir = "../dataset/result/demand_classification.csv", data = None):
    #Load phân loại deamand đã có từ trước
    df_demandType = pd.read_csv(demand_dir)

    # Merge df with df_demandType to add 'Category' column
    df = data

    # Merge df with df_demandType to add 'Category' column
    df = df.merge(df_demandType[['id', 'Category']], on='id', how='left')

    # Perform one-hot encoding on 'Category' column
    df = pd.get_dummies(df, columns=['Category'], prefix='cat')

    return df

df = add_demandFeature(data = df)

df.head()

,id,item_id,dept_id,store_id,cat_id,state_id,d,sales,date,wm_yr_wk,weekday,wday,month,year,event_name_1,event_type_1,event_name_2,event_type_2,snap_CA,snap_TX,snap_WI,sell_price,cat_Erratic (Thất thường),cat_Intermittent (Ngắt quãng),cat_Lumpy (Cục bộ),cat_Smooth (Đều đặn)
0,HOBBIES_1_002_CA_1_validation,1,0,0,0,0,d_350,0.0,2012-01-13,11150,0,7,1,2012,0,0,0,0,0.0,1.0,0.0,3.97,False,True,False,False
1,HOBBIES_1_004_CA_1_validation,3,0,0,0,0,d_350,2.0,2012-01-13,11150,0,7,1,2012,0,0,0,0,0.0,1.0,0.0,4.34,False,True,False,False
2,HOBBIES_1_005_CA_1_validation,4,0,0,0,0,d_350,0.0,2012-01-13,11150,0,7,1,2012,0,0,0,0,0.0,1.0,0.0,2.48,False,True,False,False
3,HOBBIES_1_008_CA_1_validation,7,0,0,0,0,d_350,0.0,2012-01-13,11150,0,7,1,2012,0,0,0,0,0.0,1.0,0.0,0.50,False,False,True,False
4,HOBBIES_1_009_CA_1_validation,8,0,0,0,0,d_350,2.0,2012-01-13,11150,0,7,1,2012,0,0,0,0,0.0,1.0,0.0,1.77,False,True,False,False


In [6]:
#Làm giàu đặc trưng
create_fea(df ,lags_fe = [7, 28, 56], wins_fe = [7, 28], std_fe = [28], 
           price_lags_fe = [7], price_wins_fe = [28], price_std_fe = [28])
df.shape

(40718219, 44)

In [7]:
#Peview một số dòng đầu
df.tail()

,id,item_id,dept_id,store_id,cat_id,state_id,d,sales,date,wm_yr_wk,weekday,wday,month,year,event_name_1,event_type_1,event_name_2,event_type_2,snap_CA,snap_TX,snap_WI,sell_price,cat_Erratic (Thất thường),cat_Intermittent (Ngắt quãng),cat_Lumpy (Cục bộ),cat_Smooth (Đều đặn),lag_7,lag_28,lag_56,rmean_7_7,rmean_28_7,rmean_56_7,rmean_7_28,rmean_28_28,rmean_56_28,rstd_7_28,rstd_28_28,rstd_56_28,price_lag_7,price_rmean_7_28,price_rstd_7_28,week,quarter,mday
40718214,FOODS_3_823_WI_3_validation,3044,6,9,2,2,d_1913,1.0,2016-04-24,11613,3,2,4,2016,0,0,0,0,0.0,0.0,0.0,2.98,False,True,False,False,0.0,0.0,1.0,0.571429,0.000000,0.285714,0.142857,0.250000,0.750000,0.524531,0.585314,1.109721,2.98,2.980000,0.000000e+00,16,2,24
40718215,FOODS_3_824_WI_3_validation,3045,6,9,2,2,d_1913,0.0,2016-04-24,11613,3,2,4,2016,0,0,0,0,0.0,0.0,0.0,2.48,False,True,False,False,0.0,0.0,0.0,0.285714,0.000000,0.000000,0.285714,0.000000,0.000000,0.534522,0.000000,0.000000,2.48,2.274286,2.418973e-01,16,2,24
40718216,FOODS_3_825_WI_3_validation,3046,6,9,2,2,d_1913,0.0,2016-04-24,11613,3,2,4,2016,0,0,0,0,0.0,0.0,0.0,3.98,False,True,False,False,0.0,1.0,0.0,1.000000,0.714286,0.428571,0.928571,1.250000,0.642857,1.184110,0.927961,0.731021,3.98,3.980000,5.384158e-08,16,2,24
40718217,FOODS_3_826_WI_3_validation,3047,6,9,2,2,d_1913,3.0,2016-04-24,11613,3,2,4,2016,0,0,0,0,0.0,0.0,0.0,1.28,False,True,False,False,1.0,4.0,0.0,0.714286,1.571429,0.000000,1.035714,1.250000,0.892857,1.070899,1.075829,1.165532,1.28,1.280000,0.000000e+00,16,2,24
40718218,FOODS_3_827_WI_3_validation,3048,6,9,2,2,d_1913,0.0,2016-04-24,11613,3,2,4,2016,0,0,0,0,0.0,0.0,0.0,1.00,False,True,False,False,0.0,5.0,0.0,0.000000,2.428571,1.285714,1.678571,1.964286,1.571429,1.925532,1.752927,1.989390,1.00,1.000000,0.000000e+00,16,2,24


In [8]:
#Drop các dòng không có dữ liệU
df.dropna(inplace = True)
df.shape

(38187637, 44)

In [9]:
#Các cột phân loại
cat_feats = ['item_id', 'dept_id','store_id', 'cat_id', 'state_id'] + \
["event_name_1", "event_name_2", "event_type_1", "event_type_2"] + \
['cat_Erratic (Thất thường)', 'cat_Intermittent (Ngắt quãng)', 'cat_Lumpy (Cục bộ)', 'cat_Smooth (Đều đặn)']
#Các cột không sử dụng
useless_cols = ["id", "date", "sales","d", "wm_yr_wk", "weekday"]

#Định nghĩa param cho thuật toán LightGBM
params = {
        "objective" : "poisson",
        "metric" :"rmse",
        "force_row_wise" : True,
        "learning_rate" : 0.075,
#         "sub_feature" : 0.8,
        "sub_row" : 0.75,
        "bagging_freq" : 1,
        "lambda_l2" : 0.1,
         "nthread" : 8,
        "metric": ["rmse"],
    'verbosity': 1,
    'num_iterations' : 4000,
    'num_leaves': 128,
    "min_data_in_leaf": 100,
    "device": "cpu",
    "early_stopping_round": 100
}

#Các cột sử dụng trong tập train (ngoại trừ những cột ko dùng ĐN ở trên)
train_cols = df.columns[~df.columns.isin(useless_cols)]

In [10]:
i = 0
for store, df_store in df.groupby("store_id"):

    print(f"Processing store {store}...")
    X_train = df_store[train_cols]
    y_train = df_store["sales"]

    #Cố định seed random
    np.random.seed(777)

    print(f"Create fake valid data for store {store}...")
    #Fake ra 30 000 index cho tập fake valid
    fake_valid_inds = np.random.choice(X_train.index.values, 30000, replace = False)

    #Index còn lại làm cho tập train
    train_inds = np.setdiff1d(X_train.index.values, fake_valid_inds)

    #Tập dữ liệu train
    train_data = lgb.Dataset(X_train.loc[train_inds] , label = y_train.loc[train_inds], 
                            categorical_feature=cat_feats, free_raw_data=False)

    #Tập dữ liệU valid fake
    fake_valid_data = lgb.Dataset(X_train.loc[fake_valid_inds], label = y_train.loc[fake_valid_inds],
                                categorical_feature=cat_feats,
                    free_raw_data=False)
    
    print(f"Done creating fake valid data for store {store}...")

    #Giải phóng bộ nhớ
    if i == 0:
        del df, X_train, y_train, fake_valid_inds,train_inds ; gc.collect()
        i += 1
    else:
        del X_train, y_train, fake_valid_inds,train_inds ; gc.collect()

    print(f"Training model for store {store}...")
    #Train mô hình
    m_lgb = lgb.train(params, train_data, valid_sets = [fake_valid_data],  callbacks=[lgb.log_evaluation(20)])

    #Save mô hình
    m_lgb.save_model(f"../dataset/result/model/model_store_{store}.lgb")

Processing store 0...
Create fake valid data for store 0...
Done creating fake valid data for store 0...
Training model for store 0...
[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Total Bins 6685
[LightGBM] [Info] Number of data points in the train set: 3852898, number of used features: 36
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Start training from score 0.445571
Training until validation scores don't improve for 100 rounds
[20]	valid_0's rmse: 3.79839
[40]	valid_0's rmse: 3.52815
[60]	valid_0's rmse: 3.45871
[80]	valid_0's rmse: 3.42854
[100]	valid_0's rmse: 3.41511
[120]	valid_0's rmse: 3.40663
[140]	valid_0's rmse: 3.39968
[160]	valid_0's rmse: 3.

In [11]:
alphas = [1.028, 1.023, 1.018]
weights = [1 / len(alphas)] * len(alphas)
sub = 0.

# Load toàn bộ model trước (rất quan trọng để tăng tốc)
models = {
    store_id: lgb.Booster(model_file=f"../dataset/result/model/model_store_{store_id}.lgb")
    for store_id in range(10)
}

cols = [f"d_{tr_last + i}" for i in range(1, 29)]

for icount, (alpha, weight) in enumerate(zip(alphas, weights)):

    te = create_dt(
        is_train=False,
        price_dt=PRICE_DTYPES,
        cal_dt=CAL_DTYPES,
        tr_last=tr_last
    )

    te = add_demandFeature(data=te)

    for tdelta in range(28):
        day = fday + timedelta(days=tdelta)
        print(f"Day {tdelta}: {day}")

        for store_id, model in models.items():

            # Filter data theo store ngay từ đầu → nhanh hơn rất nhiều
            tst = te[
                (te["store_id"] == store_id) &
                (te["date"] >= day - timedelta(days=max_lags)) &
                (te["date"] <= day)
            ].copy()

            if tst.empty:
                continue

            create_fea(tst,lags_fe = [7, 28, 56], wins_fe = [7, 28], std_fe = [28], 
                price_lags_fe = [7], price_wins_fe = [28], price_std_fe = [28])

            tst_day = tst.loc[tst["date"] == day, train_cols]

            if tst_day.empty:
                continue

            te.loc[
                (te["store_id"] == store_id) & (te["date"] == day),
                "sales"
            ] = alpha * model.predict(tst_day)

    # ===== reshape submission =====
    te_sub = te.loc[te.date >= fday, ["id", "sales"]].copy()

    te_sub["F"] = [
        f"d_{tr_last + rank}"
        for rank in te_sub.groupby("id").cumcount() + 1
    ]

    te_sub = te_sub.set_index(["id", "F"]).unstack()["sales"][cols].reset_index()
    te_sub.fillna(0., inplace=True)
    te_sub.sort_values("id", inplace=True)
    te_sub.reset_index(drop=True, inplace=True)

    te_sub.to_csv(f"../dataset/result/stores/submission_all_{icount}.csv", index=False)

    # ===== ensemble =====
    if icount == 0:
        sub = te_sub.copy()
        sub[cols] *= weight
    else:
        sub[cols] += te_sub[cols] * weight

    print(f"Done alpha {alpha}")

# ===== final submission =====
sub["id"] = sub["id"].str.extract(r"^(.+?)_(?:validation|evaluation)$", expand=False)
sub.columns.name = None
sub.set_index("id", inplace=True)
sub.to_csv("../dataset/result/stores/submission_all.csv", index=False)

Day 0: 2016-04-25 00:00:00
Day 1: 2016-04-26 00:00:00
Day 2: 2016-04-27 00:00:00
Day 3: 2016-04-28 00:00:00
Day 4: 2016-04-29 00:00:00
Day 5: 2016-04-30 00:00:00
Day 6: 2016-05-01 00:00:00
Day 7: 2016-05-02 00:00:00
Day 8: 2016-05-03 00:00:00
Day 9: 2016-05-04 00:00:00
Day 10: 2016-05-05 00:00:00
Day 11: 2016-05-06 00:00:00
Day 12: 2016-05-07 00:00:00
Day 13: 2016-05-08 00:00:00
Day 14: 2016-05-09 00:00:00
Day 15: 2016-05-10 00:00:00
Day 16: 2016-05-11 00:00:00
Day 17: 2016-05-12 00:00:00
Day 18: 2016-05-13 00:00:00
Day 19: 2016-05-14 00:00:00
Day 20: 2016-05-15 00:00:00
Day 21: 2016-05-16 00:00:00
Day 22: 2016-05-17 00:00:00
Day 23: 2016-05-18 00:00:00
Day 24: 2016-05-19 00:00:00
Day 25: 2016-05-20 00:00:00
Day 26: 2016-05-21 00:00:00
Day 27: 2016-05-22 00:00:00
Done alpha 1.028
Day 0: 2016-04-25 00:00:00
Day 1: 2016-04-26 00:00:00
Day 2: 2016-04-27 00:00:00
Day 3: 2016-04-28 00:00:00
Day 4: 2016-04-29 00:00:00
Day 5: 2016-04-30 00:00:00
Day 6: 2016-05-01 00:00:00
Day 7: 2016-05-02 00

In [12]:
def rmsse(train, test, forecast):

    #Sort index để phòng bị sai thứ tự
    test = test.sort_index()
    forecast = forecast.sort_index()
    train = train.sort_index()

    #Tính toán RMSSE
    forecast_mse = np.mean((test - forecast)**2, axis=1)
    train_mse = train.apply(
        lambda row: np.mean(np.diff(np.trim_zeros(row.values)) ** 2)
        if len(np.trim_zeros(row.values)) > 1 else 0,
        axis=1
    )
    return np.sqrt(forecast_mse / train_mse)

In [13]:
#Preview dữ liệu forecast
sub.head()

,d_1914,d_1915,d_1916,d_1917,d_1918,d_1919,d_1920,d_1921,d_1922,d_1923,d_1924,d_1925,d_1926,d_1927,d_1928,d_1929,d_1930,d_1931,d_1932,d_1933,d_1934,d_1935,d_1936,d_1937,d_1938,d_1939,d_1940,d_1941
id,,,,,,,,,,,,,,,,,,,,,,,,,,,,
FOODS_1_001_CA_1,0.922822,0.811001,0.822824,0.785904,0.915815,1.328313,1.246389,1.024716,0.962524,0.949702,0.938302,1.153118,1.421218,1.191220,1.028447,0.950685,0.975551,0.960702,1.142417,1.401162,1.276668,0.969564,0.903751,0.909112,0.931164,1.147825,1.397536,1.284290
FOODS_1_001_CA_2,0.974830,0.806171,0.757667,0.822066,1.033090,1.311104,1.364822,0.822650,0.901199,0.853317,1.086894,0.977091,1.203530,1.378682,0.853421,0.948855,0.896875,0.975631,1.214356,1.480741,1.665203,1.005878,1.081415,1.075092,1.297393,1.287960,1.757292,1.565312
FOODS_1_001_CA_3,0.973566,0.941995,0.754768,0.730472,1.112594,1.156751,0.950512,0.571255,0.699433,0.586315,0.670192,0.962472,1.083832,1.074155,0.823463,0.794522,0.672456,0.668186,0.996177,1.127486,1.340948,0.868287,0.806224,0.699140,0.702269,1.003697,1.099529,1.445079
FOODS_1_001_CA_4,0.360599,0.327967,0.331218,0.331218,0.322565,0.369860,0.439526,0.394384,0.378459,0.387074,0.370291,0.361919,0.417021,0.369896,0.393090,0.366570,0.372903,0.372903,0.337187,0.412592,0.409311,0.398454,0.367001,0.375279,0.385996,0.351403,0.394941,0.392569
FOODS_1_001_TX_1,0.566695,0.582142,0.573857,0.565571,0.303549,0.246318,0.263837,0.497197,0.506626,0.525762,0.404583,0.607421,0.593476,0.426943,0.531234,0.460826,0.420943,0.437724,0.420039,0.446750,0.506627,0.445833,0.410816,0.429133,0.445827,0.449170,0.532910,0.493399


In [14]:
#Xử lý lại tập test để đánh giá
df_valid = pd.read_csv("../dataset/raw/sales_train_evaluation.csv")
cols = [f"d_{i}" for i in range(tr_last + 1, 1942)]

df_valid["id"] = df_valid["id"].str.extract(r"^(.+?)_(?:validation|evaluation)$", expand=False)
df_valid.set_index("id", inplace = True)

test = df_valid[cols]

test.head()

,d_1914,d_1915,d_1916,d_1917,d_1918,d_1919,d_1920,d_1921,d_1922,d_1923,d_1924,d_1925,d_1926,d_1927,d_1928,d_1929,d_1930,d_1931,d_1932,d_1933,d_1934,d_1935,d_1936,d_1937,d_1938,d_1939,d_1940,d_1941
id,,,,,,,,,,,,,,,,,,,,,,,,,,,,
HOBBIES_1_001_CA_1,0,0,0,2,0,3,5,0,0,1,1,0,2,1,2,2,1,0,2,4,0,0,0,0,3,3,0,1
HOBBIES_1_002_CA_1,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,2,1,1,0,0,0,0,0
HOBBIES_1_003_CA_1,0,0,1,1,0,2,1,0,0,0,0,2,1,3,0,0,1,0,1,0,2,0,0,0,2,3,0,1
HOBBIES_1_004_CA_1,0,0,1,2,4,1,6,4,0,0,0,2,2,4,2,1,1,1,1,1,0,4,0,1,3,0,2,6
HOBBIES_1_005_CA_1,1,0,2,3,1,0,3,2,3,1,1,3,2,3,2,2,2,2,0,0,0,2,1,0,0,2,1,0


In [15]:
# 1. Đổi đuôi id
train = df_valid.filter(like="d_")
train.head()

,d_1,d_2,d_3,d_4,d_5,d_6,d_7,d_8,d_9,d_10,d_11,d_12,d_13,d_14,d_15,d_16,d_17,d_18,d_19,d_20,d_21,d_22,d_23,d_24,d_25,...,d_1917,d_1918,d_1919,d_1920,d_1921,d_1922,d_1923,d_1924,d_1925,d_1926,d_1927,d_1928,d_1929,d_1930,d_1931,d_1932,d_1933,d_1934,d_1935,d_1936,d_1937,d_1938,d_1939,d_1940,d_1941
id,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
HOBBIES_1_001_CA_1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,2,0,3,5,0,0,1,1,0,2,1,2,2,1,0,2,4,0,0,0,0,3,3,0,1
HOBBIES_1_002_CA_1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,2,1,1,0,0,0,0,0
HOBBIES_1_003_CA_1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,1,0,2,1,0,0,0,0,2,1,3,0,0,1,0,1,0,2,0,0,0,2,3,0,1
HOBBIES_1_004_CA_1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,2,4,1,6,4,0,0,0,2,2,4,2,1,1,1,1,1,0,4,0,1,3,0,2,6
HOBBIES_1_005_CA_1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,3,1,0,3,2,3,1,1,3,2,3,2,2,2,2,0,0,0,2,1,0,0,2,1,0


In [16]:
# Tỷ trọng
item_share = pd.read_csv("../dataset/result/item_share.csv")
item_share.set_index("item_id", inplace = True)

item_share.tail()

,sales_share
item_id,
HOBBIES_2_025_TX_1,8.218128e-08
HOUSEHOLD_1_512_CA_4,6.586242e-08
FOODS_1_079_CA_2,3.263771e-08
FOODS_1_079_CA_3,1.631885e-08
FOODS_2_394_TX_3,0.000000e+00


In [17]:
rmsse_result = rmsse(train, test, sub)


In [18]:
wrmsse = rmsse_result * item_share['sales_share']
wrmsse.sum()

np.float64(0.7769053796444829)